# 12 · Estrutura secundária (DSSP) da simulação contra o cristal

| | |
|---|---|
| **Autor** | Patrick Allan dos Santos Faustino |
| **Projeto** | Revisão tutorial de dinâmica molecular — xilanase GH11 (PDB 1XNB) |
| **Versão** | v2026.11 — notebook autocontido |

**Pergunta.** A simulação preserva a estrutura secundária do cristal 1XNB — a folha-β do enovelamento β-jelly roll das GH11 e a única hélice α?

DSSP (Kabsch & Sander, *Biopolymers* **1983**, 22, 2577), na implementação do MDTraj (baseada no
DSSP 2.2.0), em três classes: **H** (hélice: α, 3₁₀, π), **E** (folha: fita e ponte β) e **C**
(coil). A posição do H amida é **inferida** do esqueleto, como no DSSP original — então cristal
(sem H) e trajetória são classificados exatamente da mesma forma.

| Observável | Definição |
|---|---|
| Folha-β (%), Hélice (%) | fração dos resíduos em E e em H, frame a frame |
| Concordância (%) | fração dos resíduos com a **mesma** classe do cristal, frame a frame |
| Por resíduo | fração do tempo de cada resíduo no estado do cristal |

> Fitas de borda e voltas trocam de classe com facilidade (E ↔ C): concordância de 100 % não é o
> esperado em solução. O que importa é o núcleo da folha se manter.

**Blocos:** 1 bibliotecas · 2 configuração e dados · 3 cálculos · 4 estatística ·
5 gráfico (matplotlib) · 6 gráfico interativo (Plotly) · 7 exportação.

## 1 · Bibliotecas

In [ ]:
import sys
import platform
from datetime import datetime
from pathlib import Path
import re

import numpy as np
import pandas as pd
import scipy
from scipy import stats
import matplotlib
import matplotlib.pyplot as plt
from matplotlib import font_manager
import plotly
import plotly.graph_objects as go
from plotly.subplots import make_subplots
import mdtraj as md
import MDAnalysis as mda
pd.set_option("display.width", 160)
pd.set_option("display.max_columns", 40)

INICIO = datetime.now()
VERSOES = {
    "Python": platform.python_version(),
    "NumPy": np.__version__,
    "pandas": pd.__version__,
    "SciPy": scipy.__version__,
    "Matplotlib": matplotlib.__version__,
    "Plotly": plotly.__version__,
    "MDTraj": md.__version__,
    "MDAnalysis": mda.__version__,
}
print(f"Data e hora da execução: {INICIO:%d/%m/%Y %H:%M:%S}")
print(f"Sistema operacional: {platform.platform()}")
df_versoes = pd.DataFrame(list(VERSOES.items()), columns=["biblioteca", "versao"])
df_versoes

## 2 · Configuração da análise e carregamento dos dados

In [ ]:
# ── Onde estão as simulações ────────────────────────────────────────────────
PASTA_BASE = Path("..")             # pasta que contém rep1/, rep2/, ...   <- AJUSTE
SISTEMA = "xilanase_1XNB"           # nome da pasta de resultados
REPLICAS = [1, 2, 3, 4, 5]          # uma réplica só também funciona: [1]
PREFIXO_MD = "md_300ns"             # -deffnm da produção


def pasta_replica(n):
    """Pasta da réplica n."""
    return PASTA_BASE / f"rep{n}"

ARQUIVO_TOP = f"{PREFIXO_MD}.gro"            # sistema inteiro
ARQUIVO_TRAJ = f"{PREFIXO_MD}_mol.xtc"        # sistema inteiro, -pbc mol -center
ARQUIVO_TPR = f"{PREFIXO_MD}.tpr"            # elementos pelo nº atômico (MDAnalysis >= 2.11); None = pelo nome
ARQUIVOS = {"topologia": ARQUIVO_TOP, "trajetória": ARQUIVO_TRAJ, **({"tpr": ARQUIVO_TPR} if ARQUIVO_TPR else {})}

# ── Amostragem ──────────────────────────────────────────────────────────────
DT_ANALISE_PS = 100.0     # intervalo entre frames analisados
DESCARTE_NS = 0.0         # equilibração descartada no início de cada réplica (decida no 02)
NIVEL_CONFIANCA = 0.95    # intervalo de confiança (t de Student)

# ── Análise ─────────────────────────────────────────────────────────────────
SELECAO = "protein and backbone"     # o DSSP usa só N, CA, C, O
PDB_CRISTAL = PASTA_BASE / "protein" / "1XNB.pdb"
DESLOCAMENTO_RESIDUO = 0             # somado ao nº do resíduo do PDB antes de casar
OBS_E, OBS_H, OBS_C = "Folha-β", "Hélice", "Concordância com o cristal"

# ── Saída ───────────────────────────────────────────────────────────────────
DIR_SAIDA = Path("resultados") / SISTEMA / "12_dssp_cristal"
PREFIXO = "dssp"
TITULO = "12 · DSSP × cristal"

In [ ]:
linhas = []
for n in REPLICAS:
    for papel, nome in ARQUIVOS.items():
        caminho = pasta_replica(n) / nome
        linhas.append({"replica": f"rep{n}", "arquivo": papel, "caminho": str(caminho),
                       "existe": caminho.exists(),
                       "tamanho_MB": round(caminho.stat().st_size / 1024 ** 2, 1) if caminho.exists() else np.nan})
df_arquivos = pd.DataFrame(linhas)
display(df_arquivos)
if not df_arquivos["existe"].all():
    raise FileNotFoundError("Arquivos ausentes (veja a tabela acima). Ajuste RAIZ/SISTEMA/REPLICAS.")

In [ ]:
MAPA_RESNAMES = {"HIE": "HIS", "HID": "HIS", "HIP": "HIS", "HSD": "HIS", "HSE": "HIS",
                 "HSP": "HIS", "CYX": "CYS", "CYM": "CYS", "LYN": "LYS", "ASH": "ASP",
                 "GLH": "GLU"}


def tpr_de(n):
    """Caminho do .tpr da réplica n (None se ARQUIVO_TPR = None)."""
    return pasta_replica(n) / ARQUIVO_TPR if ARQUIVO_TPR else None


def elementos_do_tpr(tpr, top, arquivo):
    """Elemento de cada átomo da topologia, pelo número atômico gravado no .tpr.

    O .gro pode ser o sistema inteiro ou um subconjunto na MESMA ordem (o _prot.gro).
    O casamento é resíduo a resíduo, pela sequência de nomes de átomos: o MDAnalysis
    renumera os resíduos do .tpr a partir de 1, então o nº do resíduo (Leu4...) não serve.
    """
    u = mda.Universe(str(tpr))
    if not hasattr(u.atoms, "elements"):
        raise ValueError("o .tpr não trouxe números atômicos")
    if str(arquivo).endswith(".gro"):            # nomes crus (o MDTraj renomeia OW -> O etc.)
        linhas = Path(arquivo).read_text().splitlines()[2:2 + top.n_atoms]
        nomes = [l[10:15].strip() for l in linhas]
    else:
        nomes = [a.name for a in top.atoms]
    cortes = np.flatnonzero(np.diff(u.atoms.resindices)) + 1
    residuos_tpr = list(zip(np.split(u.atoms.names, cortes), np.split(u.atoms.elements, cortes)))
    elementos, j = [], 0
    for residuo in top.residues:
        alvo = [nomes[a.index] for a in residuo.atoms]
        while j < len(residuos_tpr) and list(residuos_tpr[j][0]) != alvo:
            j += 1
        if j == len(residuos_tpr):
            raise ValueError(f"{residuo} não casa com o .tpr (o .gro não é um subconjunto ordenado dele)")
        elementos += list(residuos_tpr[j][1])
        j += 1
    return elementos


def ler_topologia(arquivo, tpr=None):
    """Topologia MDTraj de um .gro/.pdb, com os elementos certos.

    * Com `tpr` (MDAnalysis >= 2.11 lê o .tpr do GROMACS 2026): o elemento vem do número
      atômico gravado no .tpr — a mesma informação que o GROMACS usa.
    * Sem `tpr` (ou se a leitura falhar): o MDTraj deduz o elemento pelo nome do átomo, e o
      FE do heme viraria flúor (19 u em vez de 55,85 u); ele é corrigido e um aviso é impresso.
    * Resíduos com nome de protonação (HIE, CYM...) recebem o nome padrão, para que
      `protein` e `resname` funcionem.
    """
    top = md.load_topology(str(arquivo))
    for residuo in top.residues:
        residuo.name = MAPA_RESNAMES.get(residuo.name.upper(), residuo.name)
    if tpr is not None and Path(tpr).exists():
        try:
            elementos = elementos_do_tpr(tpr, top, arquivo)
            for atomo, simbolo in zip(top.atoms, elementos):
                atomo.element = md.element.get_by_symbol(simbolo.capitalize()) if simbolo else md.element.virtual
            return top
        except Exception as erro:
            print(f"[aviso] .tpr não usado ({erro}); elementos deduzidos pelo nome do átomo.")
    for atomo in top.atoms:
        if atomo.residue.name == "HEM" and atomo.name.upper() == "FE":
            atomo.element = md.element.iron
    return top


def selecionar(top, consulta, n=None):
    """top.select com erro explícito se vazio (ou se não tiver n átomos)."""
    idx = top.select(consulta)
    if len(idx) == 0:
        raise ValueError(f"Seleção vazia: '{consulta}' (no MDTraj o nº do resíduo é `resSeq`).")
    if n is not None and len(idx) != n:
        raise ValueError(f"'{consulta}' casou com {len(idx)} átomos (esperado {n}).")
    return idx


def intervalo_do_arquivo_ps(xtc, top):
    """Δt entre frames gravados no .xtc (lê só os 2 primeiros frames)."""
    for lote in md.iterload(str(xtc), top=top, chunk=2, atom_indices=[0]):
        return float(np.diff(lote.time)[0])


def estride(xtc, top, dt_alvo_ps):
    """Passo de leitura que leva o Δt do arquivo ao Δt de análise."""
    dt = intervalo_do_arquivo_ps(xtc, top)
    passo = max(1, int(round(dt_alvo_ps / dt)))
    if dt_alvo_ps < dt:
        print(f"[aviso] Δt pedido {dt_alvo_ps:g} ps é menor que o do arquivo ({dt:g} ps).")
    elif abs(passo * dt - dt_alvo_ps) > 1e-6 * dt_alvo_ps:
        print(f"[aviso] Δt {dt_alvo_ps:g} ps não é múltiplo de {dt:g} ps; usando {passo * dt:g} ps.")
    return passo, dt


def producao(t_ns):
    """Máscara dos frames de produção (depois de DESCARTE_NS)."""
    t_ns = np.asarray(t_ns, dtype=float)
    mascara = t_ns >= t_ns[0] + DESCARTE_NS
    if not mascara.any():
        raise ValueError("DESCARTE_NS é maior que a trajetória.")
    return mascara


def carregar(n, atomos=None, dt_ps=None):
    """Lê a réplica n (só os átomos `atomos`, índices da topologia) -> (traj, t_ns)."""
    xtc, top = pasta_replica(n) / ARQUIVO_TRAJ, ler_topologia(pasta_replica(n) / ARQUIVO_TOP, tpr_de(n))
    passo, dt = estride(xtc, top, dt_ps or DT_ANALISE_PS)
    traj = md.load(str(xtc), top=top, stride=passo,
                   atom_indices=None if atomos is None else np.unique(atomos))
    t_ns = np.asarray(traj.time, dtype=float) / 1000.0
    if not np.all(np.diff(t_ns) > 0):
        raise ValueError(f"rep{n}: tempos não crescentes (use `gmx trjcat -cat`).")
    return traj, t_ns



top = ler_topologia(pasta_replica(REPLICAS[0]) / ARQUIVO_TOP, tpr_de(REPLICAS[0]))
trajs = {f"rep{n}": carregar(n, selecionar(top, SELECAO)) for n in REPLICAS}
top_bb = next(iter(trajs.values()))[0].topology
residuos_md = np.array([r.resSeq for r in top_bb.residues])

if not Path(PDB_CRISTAL).exists():
    raise FileNotFoundError(f"{PDB_CRISTAL} não existe — este notebook precisa do cristal.")
cristal = md.load_pdb(str(PDB_CRISTAL))
cristal = cristal.atom_slice(cristal.topology.select("protein and backbone"))
residuos_cr = np.array([r.resSeq for r in cristal.topology.residues]) + DESLOCAMENTO_RESIDUO
ss_cristal = pd.Series(md.compute_dssp(cristal, simplified=True)[0], index=residuos_cr)
ss_cristal = ss_cristal[~ss_cristal.index.duplicated()]
comuns = np.intersect1d(residuos_md, ss_cristal.index)
print(f"Cristal: {len(ss_cristal)} resíduos | {comuns.size} casados com a simulação | "
      f"E = {100 * (ss_cristal == 'E').mean():.1f} % | H = {100 * (ss_cristal == 'H').mean():.1f} %")
nomes_md = {r.resSeq: r.name for r in top_bb.residues}
nomes_cr = {r.resSeq + DESLOCAMENTO_RESIDUO: MAPA_RESNAMES.get(r.name, r.name) for r in cristal.topology.residues}
divergentes = [r for r in comuns if nomes_md[r] != nomes_cr[r]]
if divergentes:
    print(f"[!] {len(divergentes)} resíduos casam por número mas não por identidade (ex.: {divergentes[:5]}) — "
          f"confira DESLOCAMENTO_RESIDUO.")
df_trajetorias = pd.DataFrame([{
    "replica": rep, "frames": tr.n_frames, "atomos": tr.n_atoms,
    "dt_ps": round(1000 * float(np.median(np.diff(t))), 3),
    "inicio_ns": t[0], "fim_ns": t[-1],
    "frames_producao": int(producao(t).sum()),
    "memoria_MB": round(tr.xyz.nbytes / 1024 ** 2, 1)}
    for rep, (tr, t) in trajs.items()])
df_trajetorias

## 3 · Cálculos

In [ ]:
posicao = {r: i for i, r in enumerate(residuos_md)}
cols = np.array([posicao[r] for r in comuns])
ref = ss_cristal.loc[comuns].to_numpy()
quadros, por_residuo, mapas_ss = [], [], {}
for rep, (traj, t_ns) in trajs.items():
    ss = md.compute_dssp(traj, simplified=True)[:, cols]            # (F, resíduos comuns)
    mapas_ss[rep] = (t_ns, ss)                                       # para o mapa resíduo × tempo
    fase = np.where(producao(t_ns), "producao", "equilibracao")
    for obs, v in ((OBS_E, 100 * (ss == "E").mean(axis=1)), (OBS_H, 100 * (ss == "H").mean(axis=1)),
                   (OBS_C, 100 * (ss == ref).mean(axis=1))):
        quadros.append(pd.DataFrame({"replica": rep, "observavel": obs, "tempo_ns": t_ns, "valor": v, "fase": fase}))
    prod = ss[producao(t_ns)]
    por_residuo.append(pd.DataFrame({"replica": rep, "residuo": comuns, "nome": [nomes_md[r].capitalize() for r in comuns],
                                     "ss_cristal": ref, "frac_estado_cristal": (prod == ref).mean(axis=0),
                                     "frac_E": (prod == "E").mean(axis=0), "frac_H": (prod == "H").mean(axis=0)}))
df_series = pd.concat(quadros, ignore_index=True)
df_por_replica = pd.concat(por_residuo, ignore_index=True)
df_series.groupby(["observavel", "replica"], sort=False)["valor"].describe()

## 4 · Estatística

Séries: média, $g$, $N_{ef}$, SEM corrigido e IC por réplica; entre réplicas, média ± SEM.
Por resíduo: fração do tempo no estado do cristal, média ± SEM entre réplicas; a lista dos que
ficam abaixo de 50 % mostra **onde** a estrutura muda.

In [ ]:
def ineficiencia_estatistica(x, t_minimo=3):
    """g = 1 + 2·τ_int (em frames) — Chodera et al., JCTC 2007, 3, 26.

    ACF não enviesada (FFT), somada até o 1º C(t) <= 0 depois de t_minimo, com o
    peso (1 − t/N) aplicado UMA vez. Igual a
    pymbar.timeseries.statistical_inefficiency(fast=False).
    """
    x = np.asarray(x, dtype=float)
    n = x.size
    if n < 3 or np.allclose(x, x[0]):
        return 1.0
    d = x - x.mean()
    var = float(d @ d) / n
    tamanho = 1 << (2 * n - 1).bit_length()
    f = np.fft.rfft(d, tamanho)
    acf = np.fft.irfft(f * np.conjugate(f), tamanho)[:n] / ((n - np.arange(n)) * var)
    g = 1.0
    for t in range(1, n - 1):
        if acf[t] <= 0.0 and t > t_minimo:
            break
        g += 2.0 * acf[t] * (1.0 - t / n)
    return float(max(g, 1.0))


def t_student(graus_liberdade):
    return stats.t.ppf(0.5 + NIVEL_CONFIANCA / 2, df=max(graus_liberdade, 1))


def resumo_replica(y, dt_ns, observavel, replica, unidade=""):
    """Média, DP, g, N_ef e SEM corrigido de UMA série (uma réplica)."""
    y = np.asarray(y, dtype=float)
    g = ineficiencia_estatistica(y)
    n_ef = max(y.size / g, 2.0)
    dp = float(np.std(y, ddof=1))
    sem = dp / np.sqrt(n_ef)
    media = float(np.mean(y))
    return {"observavel": observavel, "replica": replica, "unidade": unidade,
            "n": y.size, "media": media, "desvio_padrao": dp, "sem": sem,
            "ic_inferior": media - t_student(n_ef - 1) * sem,
            "ic_superior": media + t_student(n_ef - 1) * sem,
            "minimo": float(y.min()), "maximo": float(y.max()),
            "g": g, "tau_int_ns": (g - 1.0) / 2.0 * dt_ns, "n_efetivo": n_ef}


def entre_replicas(medias, observavel, unidade=""):
    """A réplica é a unidade estatística: média ± SEM das médias das réplicas."""
    medias = np.asarray(medias, dtype=float)
    k = medias.size
    media = float(medias.mean())
    dp = float(np.std(medias, ddof=1)) if k > 1 else np.nan
    sem = dp / np.sqrt(k) if k > 1 else np.nan
    return {"observavel": observavel, "replica": "entre_replicas", "unidade": unidade,
            "n": k, "media": media, "desvio_padrao": dp, "sem": sem,
            "ic_inferior": media - t_student(k - 1) * sem if k > 1 else np.nan,
            "ic_superior": media + t_student(k - 1) * sem if k > 1 else np.nan,
            "minimo": float(medias.min()), "maximo": float(medias.max()),
            "g": np.nan, "tau_int_ns": np.nan, "n_efetivo": float(k)}


def estatisticas_series(df_series, unidades):
    """Por réplica (só produção) + linha 'entre_replicas' quando há 2 ou mais."""
    linhas = []
    prod = df_series[df_series["fase"] == "producao"]
    for obs, bloco in prod.groupby("observavel", sort=False):
        por_rep = [resumo_replica(g["valor"], float(np.median(np.diff(g["tempo_ns"]))),
                                  obs, rep, unidades.get(obs, ""))
                   for rep, g in bloco.groupby("replica", sort=False)]
        linhas += por_rep
        if len(por_rep) > 1:
            linhas.append(entre_replicas([l["media"] for l in por_rep], obs, unidades.get(obs, "")))
    return pd.DataFrame(linhas)


def referencia(df_stats, observavel):
    """Linha 'entre_replicas' do observável (ou a da única réplica)."""
    bloco = df_stats[df_stats["observavel"] == observavel]
    entre = bloco[bloco["replica"] == "entre_replicas"]
    return (entre if len(entre) else bloco).iloc[0]


def formatar(media, erro):
    """'média ± erro' com 2 algarismos significativos no erro (GUM, JCGM 100:2008 §7.2.6)."""
    if not (np.isfinite(erro) and erro > 0):
        return f"{media:.4g}"
    casas = max(0, 1 - int(np.floor(np.log10(abs(erro)))))
    return f"{media:.{casas}f} ± {erro:.{casas}f}"


_SOBRESCRITO = str.maketrans("⁰¹²³⁴⁵⁶⁷⁸⁹", "0123456789")


def unidade_mpl(unidade):
    """Unidade para o matplotlib: '⁻³' vira mathtext ($^{-3}$). O sinal sobrescrito '⁻' (U+207B)
    falta em várias fontes (a Aptos Serif, inclusive) e sai como um quadrado na figura."""
    return re.sub(r"⁻([⁰¹²³⁴⁵⁶⁷⁸⁹]+)",
                  lambda m: "$^{-" + m.group(1).translate(_SOBRESCRITO) + "}$", unidade or "")


def rotulo_media(ref):
    """Rótulo dos painéis: 'média ± EP (N réplicas): valor ± erro unidade'.

    EP = erro padrão da média: entre as médias das réplicas (a réplica é a unidade estatística);
    com uma réplica só, o SEM da série corrigido pela ineficiência estatística."""
    n = ref.get("n", np.nan)
    origem = (f"{int(n)} réplicas" if ref.get("replica") == "entre_replicas" and pd.notna(n)
              else "1 réplica")
    return (f"média ± EP ({origem}): {formatar(ref['media'], ref['sem'])} "
            f"{unidade_mpl(ref.get('unidade', ''))}").strip()

In [ ]:
df_stats = estatisticas_series(df_series, {OBS_E: "%", OBS_H: "%", OBS_C: "%"})
df_cristal = pd.DataFrame([{"observavel": OBS_E, "cristal_pct": 100 * (ref == "E").mean()},
                           {"observavel": OBS_H, "cristal_pct": 100 * (ref == "H").mean()},
                           {"observavel": OBS_C, "cristal_pct": 100.0}])
tab = df_por_replica.pivot(index="residuo", columns="replica", values="frac_estado_cristal")
k = tab.shape[1]
df_residuos = (df_por_replica.groupby("residuo", sort=True)[["nome", "ss_cristal"]].first()
               .assign(frac_estado_cristal=tab.mean(axis=1),
                       sem=tab.std(axis=1, ddof=1) / np.sqrt(k) if k > 1 else np.nan,
                       frac_E=df_por_replica.groupby("residuo")["frac_E"].mean(),
                       frac_H=df_por_replica.groupby("residuo")["frac_H"].mean())
               .reset_index())
df_residuos["ss_dm_consenso"] = np.select([df_residuos["frac_E"] >= 0.5, df_residuos["frac_H"] >= 0.5], ["E", "H"], "C")
if len(REPLICAS) == 1:
    print("[1 réplica] Não há linha 'entre_replicas': o erro é o SEM da própria série, "
          "corrigido pela ineficiência estatística (subestima a incerteza de observáveis lentos).")
instaveis = df_residuos[df_residuos["frac_estado_cristal"] < 0.5]
print(f"{len(instaveis)} resíduo(s) passam menos da metade do tempo no estado do cristal:")
display(instaveis[["residuo", "nome", "ss_cristal", "frac_estado_cristal", "frac_E", "frac_H"]])
tabela = df_stats[df_stats["replica"] == "entre_replicas"] if len(REPLICAS) > 1 else df_stats
tabela[["observavel", "replica", "unidade", "media", "sem", "ic_inferior", "ic_superior", "n_efetivo"]].merge(
    df_cristal, on="observavel").assign(**{"média ± SEM": lambda d: [formatar(m, s) for m, s in zip(d["media"], d["sem"])]})

## 5 · Gráfico (matplotlib)

**Figura 1** — % de folha, % de hélice e concordância ao longo do tempo (pontilhado = cristal).
**Figura 2** — mapa por resíduo: classe no cristal, consenso da DM e fração do tempo no estado do
cristal. **Figura 3** — o mapa clássico do DSSP, resíduo × tempo, por réplica.

In [ ]:
# ── Estilo geral (vale para todas as figuras deste notebook) ────────────────
FONTE = "Aptos Serif"                 # se não estiver instalada, usa DejaVu Serif
TAMANHO_FONTE = 8                     # pt
CORES = ["#003f5c", "#bc5090", "#ffa600", "#58508d", "#ff6361"]   # uma por réplica
COR_MEDIA = "#333333"                 # tracejado da média
COR_REFERENCIA = "#d62728"            # linhas de referência e faixa do descarte

if FONTE not in {f.name for f in font_manager.fontManager.ttflist}:
    FONTE = "DejaVu Serif"
plt.rcParams.update({
    "font.family": FONTE, "font.size": TAMANHO_FONTE,
    "axes.labelsize": TAMANHO_FONTE, "axes.titlesize": TAMANHO_FONTE,
    "xtick.labelsize": TAMANHO_FONTE - 1, "ytick.labelsize": TAMANHO_FONTE - 1,
    "legend.fontsize": TAMANHO_FONTE - 1, "axes.linewidth": 0.6,
    "xtick.major.width": 0.6, "ytick.major.width": 0.6,
    "figure.dpi": 120,
    "pdf.fonttype": 42,                # fontes embutidas como TrueType (editáveis)
    "svg.fonttype": "none",            # texto do SVG continua texto
})

### Figura 1 — séries temporais

In [ ]:
LINHAS_CRISTAL = {l.observavel: [(l.cristal_pct, "1XNB")] for l in df_cristal.itertuples() if l.observavel != OBS_C}

In [ ]:
# ── Layout desta figura ─────────────────────────────────────────────────────
LARGURA = 7.0                  # polegadas
ALTURA_LINHA = 1.6            # polegadas por observável
ESPESSURA = 0.6
OPACIDADE = 0.85
MEDIA_MOVEL_NS = 1.0           # janela da média móvel (ns); None = só a série bruta
OPACIDADE_BRUTA = 0.25         # série bruta ao fundo quando há média móvel
MOSTRAR_HISTOGRAMA = True      # distribuição da produção à direita de cada série
BINS = 40
LETRAS = "ABCDEFGHIJ"          # "" = sem letras
ROTULOS_Y = {OBS_E: "Folha-β (%)", OBS_H: "Hélice (%)", OBS_C: "Concordância (%)"}
LINHAS_REF = LINHAS_CRISTAL            # {observável: [(valor, texto), ...]}
LIMITES_Y = {}          # {observável: (mín, máx)}

# ── Gráfico ─────────────────────────────────────────────────────────────────
observaveis = list(df_series["observavel"].unique())
fig, eixos = plt.subplots(len(observaveis), 2 if MOSTRAR_HISTOGRAMA else 1,
                          figsize=(LARGURA, ALTURA_LINHA * len(observaveis)),
                          sharey="row", squeeze=False, layout="constrained",
                          gridspec_kw={"width_ratios": [4, 1]} if MOSTRAR_HISTOGRAMA else None)
for i, obs in enumerate(observaveis):
    ax = eixos[i, 0]
    dados = df_series[df_series["observavel"] == obs]
    for k, (rep, g) in enumerate(dados.groupby("replica", sort=False)):
        cor = CORES[k % len(CORES)]
        if MEDIA_MOVEL_NS:
            janela = max(1, int(round(MEDIA_MOVEL_NS / np.median(np.diff(g["tempo_ns"])))))
            ax.plot(g["tempo_ns"], g["valor"], color=cor, lw=ESPESSURA * 0.6, alpha=OPACIDADE_BRUTA)
            ax.plot(g["tempo_ns"], g["valor"].rolling(janela, center=True, min_periods=1).mean(),
                    color=cor, lw=ESPESSURA * 1.6, alpha=OPACIDADE, label=rep)
        else:
            ax.plot(g["tempo_ns"], g["valor"], color=cor, lw=ESPESSURA, alpha=OPACIDADE, label=rep)
    t0 = dados["tempo_ns"].min()
    ax.axvspan(t0, t0 + DESCARTE_NS, color=COR_REFERENCIA, alpha=0.08, lw=0)   # descarte
    ref = referencia(df_stats, obs)
    ax.axhline(ref["media"], color=COR_MEDIA, ls="--", lw=0.8)
    for valor, texto in LINHAS_REF.get(obs, []):
        ax.axhline(valor, color=COR_REFERENCIA, ls=":", lw=0.8)
        ax.annotate(texto, (1, valor), xycoords=("axes fraction", "data"), ha="right", va="bottom",
                    color=COR_REFERENCIA, fontsize=TAMANHO_FONTE - 2,
                    bbox=dict(fc="white", ec="none", alpha=0.7, pad=0.5))
    ax.set_ylabel(ROTULOS_Y.get(obs, obs))
    ax.set_title(LETRAS[i] if i < len(LETRAS) else "", loc="left", fontweight="bold")
    ax.set_title(rotulo_media(ref),
                 loc="right", fontsize=TAMANHO_FONTE - 1)
    if obs in LIMITES_Y:
        ax.set_ylim(LIMITES_Y[obs])
    ax.set_xlim(df_series["tempo_ns"].min(), df_series["tempo_ns"].max())
    if i < len(observaveis) - 1:
        ax.tick_params(labelbottom=False)
    if MOSTRAR_HISTOGRAMA:
        axh = eixos[i, 1]
        valores_prod = dados.loc[dados["fase"] == "producao", "valor"].to_numpy()
        unicos = np.unique(np.round(valores_prod, 6))
        passo = np.diff(unicos).min() if unicos.size > 1 else 0
        # porcentagens de DSSP são discretas (1 resíduo = 100/N %): um bin por valor possível, sem lacunas
        bordas = (np.arange(unicos[0] - passo / 2, unicos[-1] + passo, passo)
                  if passo > 0 and (unicos[-1] - unicos[0]) / passo <= 200 else BINS)
        axh.hist(valores_prod, bins=bordas, density=True, orientation="horizontal", color=CORES[0], alpha=0.4)
        axh.axhline(ref["media"], color=COR_MEDIA, ls="--", lw=0.8)
        axh.set_xticks([])                     # escala própria em cada linha
        axh.spines[["top", "right"]].set_visible(False)
eixos[-1, 0].set_xlabel("Tempo (ns)")
if MOSTRAR_HISTOGRAMA:
    eixos[-1, 1].set_xlabel("Distribuição")
fig.legend(*eixos[0, 0].get_legend_handles_labels(), loc="outside upper center",
           ncol=len(REPLICAS), frameon=True)
fig_series = fig
plt.show()

### Figura 2 — mapa por resíduo

In [ ]:
# ── Layout desta figura ─────────────────────────────────────────────────────
LARGURA, ALTURA = 7.0, 2.4
CORES_SS = {"H": "#5b8ff9", "E": "#f6bd16", "C": "#ffffff"}
COR_BARRA = "#003f5c"

# ── Gráfico ─────────────────────────────────────────────────────────────────
from matplotlib.colors import ListedColormap
codigo = {"C": 0, "E": 1, "H": 2}
mapa = ListedColormap([CORES_SS["C"], CORES_SS["E"], CORES_SS["H"]])
x = df_residuos["residuo"].to_numpy()
fig, (ax_ss, ax_f) = plt.subplots(2, 1, figsize=(LARGURA, ALTURA), sharex=True, layout="constrained",
                                  gridspec_kw={"height_ratios": [1, 2.2]})
faixas = np.vstack([df_residuos["ss_cristal"].map(codigo), df_residuos["ss_dm_consenso"].map(codigo)])
ax_ss.pcolormesh(np.r_[x - 0.5, x[-1] + 0.5], [0, 1, 2], faixas, cmap=mapa, vmin=0, vmax=2, edgecolors="none")
ax_ss.set(yticks=[0.5, 1.5], yticklabels=["1XNB", "DM"])
ax_f.bar(x, df_residuos["frac_estado_cristal"], yerr=df_residuos["sem"].fillna(0), width=0.9, color=COR_BARRA,
         alpha=0.8, error_kw=dict(lw=0.4))
ax_f.axhline(0.5, color=COR_REFERENCIA, ls=":", lw=0.8)
ax_f.set(xlabel="Resíduo", ylabel="fração no estado\ndo cristal", ylim=(0, 1.05), xlim=(x.min() - 0.5, x.max() + 0.5))
for nome, cor in (("hélice", CORES_SS["H"]), ("folha-β", CORES_SS["E"])):
    ax_ss.fill_between([], [], color=cor, label=nome)
ax_ss.legend(loc="lower right", bbox_to_anchor=(1, 1), ncol=2, frameon=True, fontsize=TAMANHO_FONTE - 2)
for eixo, letra in zip((ax_ss, ax_f), "AB"):
    eixo.set_title(letra, loc="left", fontweight="bold")
fig_mapa = fig
plt.show()

### Figura 3 — estrutura secundária por resíduo ao longo do tempo

O gráfico clássico do DSSP (o `.xpm` do `gmx do_dssp`/`gmx dssp`): tempo no eixo x, resíduo no
eixo y, uma cor por classe. Um painel por réplica, com a faixa do cristal à esquerda.

In [ ]:
# ── Layout desta figura ─────────────────────────────────────────────────────
LARGURA = 7.0
ALTURA_REPLICA = 1.6             # polegadas por réplica
CORES_SS = {"H": "#5b8ff9", "E": "#f6bd16", "C": "#ffffff"}

# ── Gráfico ─────────────────────────────────────────────────────────────────
from matplotlib.colors import ListedColormap
codigo = {"C": 0, "E": 1, "H": 2}
mapa = ListedColormap([CORES_SS["C"], CORES_SS["E"], CORES_SS["H"]])
para_codigo = np.vectorize(lambda c: codigo.get(c, 0))
y = np.r_[comuns - 0.5, comuns[-1] + 0.5]
fig, eixos = plt.subplots(len(mapas_ss), 2, figsize=(LARGURA, ALTURA_REPLICA * len(mapas_ss)), squeeze=False,
                          sharey=True, layout="constrained", gridspec_kw={"width_ratios": [0.04, 1]})
for i, (rep, (t_ns, ss)) in enumerate(mapas_ss.items()):
    eixos[i, 0].pcolormesh([0, 1], y, para_codigo(ss_cristal.loc[comuns].to_numpy())[:, None], cmap=mapa, vmin=0, vmax=2)
    eixos[i, 0].set(xticks=[], ylabel=f"{rep}\nresíduo")
    bordas_t = np.r_[t_ns - np.median(np.diff(t_ns)) / 2, t_ns[-1] + np.median(np.diff(t_ns)) / 2]
    eixos[i, 1].pcolormesh(bordas_t, y, para_codigo(ss).T, cmap=mapa, vmin=0, vmax=2, rasterized=True)
    if i < len(mapas_ss) - 1:
        eixos[i, 1].tick_params(labelbottom=False)
eixos[0, 0].set_title("1XNB", fontsize=TAMANHO_FONTE - 2)
eixos[-1, 1].set_xlabel("Tempo (ns)")
for nome, cor in (("hélice", CORES_SS["H"]), ("folha-β", CORES_SS["E"]), ("coil", CORES_SS["C"])):
    eixos[0, 1].fill_between([], [], facecolor=cor, edgecolor="#999999", lw=0.5, label=nome)
fig.legend(*eixos[0, 1].get_legend_handles_labels(), loc="outside upper center", ncol=3, frameon=True)
fig_tempo = fig
plt.show()

## 6 · Gráfico interativo (Plotly)

In [ ]:
# ── Layout ──────────────────────────────────────────────────────────────────
ALTURA_LINHA_PX = 220        # pixels por observável
LARGURA_PX = None              # None = largura da página
PASSO = 1                     # 1 ponto a cada N (deixa o HTML leve)
MEDIA_MOVEL_NS = 1.0           # janela da média móvel (ns); None = só a série bruta
MOSTRAR_HISTOGRAMA = True      # distribuição da produção à direita de cada série
BINS = 40
LINHAS_REF = LINHAS_CRISTAL            # {observável: [(valor, texto), ...]}
TEMPLATE = "simple_white"
TAMANHO_FONTE_PX = 13

# ── Gráfico ─────────────────────────────────────────────────────────────────
observaveis = list(df_series["observavel"].unique())
fig_i = make_subplots(rows=len(observaveis), cols=2 if MOSTRAR_HISTOGRAMA else 1, shared_yaxes=True,
                      column_widths=[0.8, 0.2] if MOSTRAR_HISTOGRAMA else None,
                      horizontal_spacing=0.02, vertical_spacing=0.05)
for i, obs in enumerate(observaveis, start=1):
    dados = df_series[df_series["observavel"] == obs]
    for k, (rep, g) in enumerate(dados.groupby("replica", sort=False)):
        cor = CORES[k % len(CORES)]
        if MEDIA_MOVEL_NS:
            janela = max(1, int(round(MEDIA_MOVEL_NS / np.median(np.diff(g["tempo_ns"])))))
            suave = g["valor"].rolling(janela, center=True, min_periods=1).mean().iloc[::PASSO]
            g = g.iloc[::PASSO]
            fig_i.add_trace(go.Scattergl(x=g["tempo_ns"], y=g["valor"], mode="lines", name=rep, legendgroup=rep,
                                         showlegend=False, opacity=0.3, line=dict(color=cor, width=0.8)),
                            row=i, col=1)
            fig_i.add_trace(go.Scattergl(x=g["tempo_ns"], y=suave, mode="lines", name=rep, legendgroup=rep,
                                         showlegend=(i == 1), line=dict(color=cor, width=1.8)), row=i, col=1)
        else:
            g = g.iloc[::PASSO]
            fig_i.add_trace(go.Scattergl(x=g["tempo_ns"], y=g["valor"], mode="lines", name=rep,
                                         legendgroup=rep, showlegend=(i == 1),
                                         line=dict(color=cor, width=1)), row=i, col=1)
    ref = referencia(df_stats, obs)
    t0 = dados["tempo_ns"].min()
    fig_i.add_vrect(x0=t0, x1=t0 + DESCARTE_NS, fillcolor=COR_REFERENCIA, opacity=0.08,
                    line_width=0, row=i, col=1)
    # média: traço WebGL (uma linha de forma ficaria escondida atrás das séries Scattergl)
    fig_i.add_trace(go.Scattergl(x=[t0, dados["tempo_ns"].max()], y=[ref["media"]] * 2, mode="lines",
                                 name="média", showlegend=False,
                                 line=dict(color=COR_MEDIA, dash="dash", width=1.5)), row=i, col=1)
    fig_i.add_annotation(x=t0, y=ref["media"], text=f"média ± EP: {formatar(ref['media'], ref['sem'])}",
                         showarrow=False, xanchor="left", yanchor="bottom",
                         bgcolor="rgba(255,255,255,0.7)", row=i, col=1)
    for valor, texto in LINHAS_REF.get(obs, []):                 # referências (cristal, literatura)
        fig_i.add_trace(go.Scattergl(x=[t0, dados["tempo_ns"].max()], y=[valor] * 2, mode="lines",
                                     name=texto, showlegend=False,
                                     line=dict(color=COR_REFERENCIA, dash="dot", width=1.2)), row=i, col=1)
        fig_i.add_annotation(x=dados["tempo_ns"].max(), y=valor, text=texto, showarrow=False,
                             xanchor="right", yanchor="bottom", font=dict(color=COR_REFERENCIA),
                             bgcolor="rgba(255,255,255,0.7)", row=i, col=1)
    fig_i.update_yaxes(title_text=f"{obs} ({ref['unidade']})" if ref["unidade"] else obs,
                       row=i, col=1)
    valores_prod = dados.loc[dados["fase"] == "producao", "valor"]
    if MOSTRAR_HISTOGRAMA and np.ptp(valores_prod) > 0:                   # série constante não tem distribuição
        fig_i.add_trace(go.Histogram(y=valores_prod, nbinsy=BINS,
                                     histnorm="probability density", marker=dict(color=CORES[0], line_width=0),
                                     opacity=0.5, name="produção", showlegend=False), row=i, col=2)
        fig_i.add_hline(y=ref["media"], line=dict(color=COR_MEDIA, dash="dash", width=1), row=i, col=2)
    if MOSTRAR_HISTOGRAMA:
        fig_i.update_xaxes(showticklabels=False, row=i, col=2)   # escala própria em cada linha
fig_i.update_xaxes(matches="x", col=1)                           # séries com o mesmo eixo de tempo
for i in range(1, len(observaveis)):
    fig_i.update_xaxes(showticklabels=False, row=i, col=1)       # rótulos do tempo só embaixo
fig_i.update_xaxes(title_text="Tempo (ns)", row=len(observaveis), col=1)
if MOSTRAR_HISTOGRAMA:
    fig_i.update_xaxes(title_text="Distribuição", row=len(observaveis), col=2)
fig_i.update_layout(template=TEMPLATE, height=ALTURA_LINHA_PX * len(observaveis) + 80,
                    width=LARGURA_PX, font=dict(family=FONTE, size=TAMANHO_FONTE_PX),
                    legend_title_text="Réplica", bargap=0)
fig_interativo = fig_i
fig_i.show()

In [ ]:
# ── Mapa interativo resíduo × tempo (uma réplica) ───────────────────────────
REPLICA_MAPA = next(iter(mapas_ss))     # ex.: "rep2"
ALTURA_PX = 420

t_ns, ss = mapas_ss[REPLICA_MAPA]
fig_i = go.Figure(go.Heatmap(x=t_ns, y=comuns, z=para_codigo(ss).T, text=ss.T, zmin=0, zmax=2, showscale=False,
                             colorscale=[[0, "#ffffff"], [0.33, "#ffffff"], [0.33, "#f6bd16"], [0.66, "#f6bd16"],
                                         [0.66, "#5b8ff9"], [1, "#5b8ff9"]],
                             hovertemplate="t = %{x:.1f} ns | resíduo %{y}: %{text}<extra></extra>"))
fig_i.update_layout(template="simple_white", height=ALTURA_PX, font=dict(family=FONTE, size=13),
                    title_text=f"{REPLICA_MAPA}: hélice (azul), folha-β (amarelo), coil (branco)",
                    xaxis_title="Tempo (ns)", yaxis_title="Resíduo")
fig_mapa_interativo = fig_i
fig_i.show()

## 7 · Exportação

Tudo vai para `DIR_SAIDA`, separado por formato:

```
{DIR_SAIDA}/
├── png/   600 dpi
├── pdf/   vetorial, fontes embutidas
├── svg/   vetorial, texto editável
├── html/  gráficos interativos (Plotly)
└── <prefixo>_dados.csv   ← CSV ÚNICO com todos os dados
```

O CSV é **um só**, em formato longo: a coluna `tabela` diz de que tabela vem
cada linha; as colunas que não se aplicam ficam vazias. As linhas iniciadas por
`#` registram data, hora, versões e parâmetros. Para ler:
`pd.read_csv(arquivo, comment="#")`.

| `tabela` | conteúdo |
|---|---|
| `serie` | % folha, % hélice e concordância com o cristal ao longo do tempo |
| `estatistica` | média, DP, SEM, IC, g, N_ef por réplica e `entre_replicas` |
| `cristal` | % folha e % hélice no 1XNB |
| `residuos` | por resíduo: classe no cristal, consenso da DM, fração no estado do cristal ± SEM |
| `residuos_por_replica` | idem, por réplica |

In [ ]:
FORMATOS = ("png", "pdf", "svg")
DPI = 600
FUNDO_TRANSPARENTE = False
PLOTLY_JS = "cdn"                  # "cdn" = HTML leve; True = embute o plotly.js (abre offline)
EXPORTAR_PLOTLY_ESTATICO = False   # True = também PNG/PDF/SVG do Plotly (precisa do kaleido)

FIGURAS = {"dssp_series": fig_series, "dssp_mapa_residuos": fig_mapa, "dssp_residuo_tempo": fig_tempo}
INTERATIVOS = {"dssp_series": fig_interativo, "dssp_residuo_tempo": fig_mapa_interativo}
TABELAS = {"serie": df_series, "estatistica": df_stats, "cristal": df_cristal, "residuos": df_residuos,
          "residuos_por_replica": df_por_replica}
PARAMETROS = {"descarte_ns": DESCARTE_NS, "dt_analise_ps": DT_ANALISE_PS, "pdb_cristal": PDB_CRISTAL,
              "deslocamento": DESLOCAMENTO_RESIDUO}

# ── Figuras ─────────────────────────────────────────────────────────────────
for nome, fig in FIGURAS.items():
    for ext in FORMATOS:
        (DIR_SAIDA / ext).mkdir(parents=True, exist_ok=True)
        fig.savefig(DIR_SAIDA / ext / f"{nome}.{ext}", dpi=DPI, bbox_inches="tight",
                    transparent=FUNDO_TRANSPARENTE)
for nome, fig in INTERATIVOS.items():
    (DIR_SAIDA / "html").mkdir(parents=True, exist_ok=True)
    fig.write_html(DIR_SAIDA / "html" / f"{nome}.html", include_plotlyjs=PLOTLY_JS,
                   config={"displaylogo": False, "toImageButtonOptions": {"format": "svg"}})
    if EXPORTAR_PLOTLY_ESTATICO:
        for ext in FORMATOS:
            try:
                fig.write_image(DIR_SAIDA / ext / f"{nome}_interativo.{ext}", scale=3)
            except Exception as erro:
                print(f"[aviso] {nome}.{ext} do Plotly não exportado: {erro}")
                break

# ── CSV único ───────────────────────────────────────────────────────────────
DIR_SAIDA.mkdir(parents=True, exist_ok=True)
partes = []
for nome, tabela in TABELAS.items():
    t = tabela.copy()
    t.insert(0, "tabela", nome)
    partes.append(t)
dados = pd.concat(partes, ignore_index=True, sort=False)
arquivo_csv = DIR_SAIDA / f"{PREFIXO}_dados.csv"
with open(arquivo_csv, "w", encoding="utf-8", newline="") as f:
    f.write(f"# {TITULO}\n")
    f.write(f"# gerado em {INICIO:%Y-%m-%d %H:%M:%S} | sistema {SISTEMA} | réplicas {REPLICAS}\n")
    f.write("# versoes: " + "; ".join(f"{k} {v}" for k, v in VERSOES.items()) + "\n")
    f.write("# parametros: " + "; ".join(f"{k} = {v}" for k, v in PARAMETROS.items()) + "\n")
    f.write("# tabelas: " + ", ".join(f"{k} ({len(v)} linhas)" for k, v in TABELAS.items()) + "\n")
    dados.to_csv(f, index=False)

# ── O que foi gravado ───────────────────────────────────────────────────────
df_exportados = pd.DataFrame([{"pasta": str(p.parent.relative_to(DIR_SAIDA)), "arquivo": p.name,
                               "tamanho_kB": round(p.stat().st_size / 1024, 1)}
                              for p in sorted(DIR_SAIDA.rglob("*")) if p.is_file()])
print(f"Resultados em: {DIR_SAIDA.resolve()}")
print(f"Duração da execução: {datetime.now() - INICIO}")
df_exportados